# Lab 4 — Build the Audience, Then Buy It
Name: Yeromenko Marharyta | Group: KM-42

## Setup

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

In [2]:
# Common objects for all tasks: the October panel, persons (R and L) with age and sex.
first_day, last_day = pd.Timestamp("2024-10-01"), pd.Timestamp("2024-10-31")
days = pd.date_range(first_day, last_day)  # the 31 broadcast days

prog = programs.copy()
prog["broadcast_date"] = pd.to_datetime(prog["broadcast_date"]).astype("datetime64[ns]")
prog["start_ts"] = prog["start_ts"].astype("datetime64[ns]")
prog["end_ts"] = prog["end_ts"].astype("datetime64[ns]")
prog["label"] = prog["program_name"] + " (" + prog["network_code"] + ")"
nfl = (prog["program_name"] == "NFL Football")
prog.loc[nfl & (prog["network_code"] == "NBC"), "label"] = "Sunday Night Football (NBC)"
prog.loc[nfl & (prog["network_code"] == "ABC"), "label"] = "Monday Night Football (ABC)"

dw = daily_weights.copy()
dw["broadcast_date"] = pd.to_datetime(dw["broadcast_date"]).astype("datetime64[ns]")
dw = dw[(dw["broadcast_date"] >= first_day) & (dw["broadcast_date"] <= last_day)]
dw_p = dw[dw["resp_type"].isin(["R", "L"])]  # persons in-tab: one row per person and day

# person viewing sessions (R and L), with the broadcast day the session starts in
v = viewings[viewings["resp_type"].isin(["R", "L"])].copy()
v["start_ts"] = v["start_ts"].astype("datetime64[ns]")
v["end_ts"] = v["end_ts"].astype("datetime64[ns]")
v["bc_date"] = (v["start_ts"] - pd.Timedelta(hours=6)).dt.normalize()

va = viewer_attributes
persons = set(va.loc[va["attr_id"].isin([101, 102]), "respondent_id"])  # R and L records
age = va[va["attr_id"] == 300].drop_duplicates("respondent_id").set_index("respondent_id")["value"]
age = age[age.index.isin(persons)]
sex = (va[va["attr_id"].isin([201, 202])].drop_duplicates("respondent_id")
       .set_index("respondent_id")["attr_id"].map({201: "M", 202: "W"}))
adults = set(age[age >= 18].index)
p2_plus = set(age[age >= 2].index)  # persons aged 2+


def daily_size(ids):
    """Sum of the daily weights of the audience's in-tab members, for each of the 31 days."""
    s = dw_p[dw_p["respondent_id"].isin(ids)].groupby("broadcast_date")["weight"].sum()
    return s.reindex(days, fill_value=0)


def size(ids):
    """Audience size: the average of the 31 daily sums."""
    return daily_size(ids).mean()


print("Persons (R, L):", len(persons), "| with an age:", len(age), "| adults 18+:", len(adults))

Persons (R, L): 10216 | with an age: 10216 | adults 18+: 8010


## Task 1.0 — Clean the survey file

In [3]:
# Look at the file before trusting it: the codebook, the first and last rows, how VALUE is written.
display(survey_layout)
display(survey.head(10))
display(survey.tail(3))
print("VALUE as delivered:", survey["VALUE"].value_counts().to_dict())

,HIERARCHY,RECTYPE,KEYWORD,Text,Variable,SAMPLE
0,1,H,H7100,Athletic shoes,APPAREL > ATHLETIC SHOES,NaN
1,1.1,H,H7101,Pairs bought in last 12 months,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,NaN
2,1.1.1,B,X710100,Any,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,4779
3,1.1.1,B,X710101,1 pair,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1810
4,1.1.1,B,X710102,2 pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1308
5,1.1.1,B,X710103,3 or more pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1661
6,1.2,H,H7102,Brands bought in last 12 months,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,NaN
7,1.2.1,B,X710201,STRIDE,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1563
8,1.2.1,B,X710202,KESTREL,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,354
9,1.2.1,B,X710203,EASYWALK,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1050


,RESPID,KEYWORD,VALUE
0,10415501,X710298,1
1,10072201,x710100,1
2,10302802,X710299,1
3,10070002,X710100,1
4,10130702,X710203,1
5,10243701,X710100,1
6,10342601,X710103,0
7,10399201,X710298,1
8,10364102,X710298,1
9,10200102,X710102,1


,RESPID,KEYWORD,VALUE
18346,10231202,X710298,1
18347,10079101,X710100,1
18348,TOTAL ROWS: 18346,,


VALUE as delivered: {'1': 16410, '1.0': 262, 'y': 246, 'TRUE': 241, '0': 237, 'Y': 228, 'yes': 217, '0.0': 129, 'N': 127, 'no': 126, '': 124, 'VALUE': 2}


In [4]:
# Answer codes from the layout
ANY = "X710100"
PAIRS = {"X710101": 1, "X710102": 2, "X710103": 3}  # 1 pair, 2 pairs, 3 or more
BRANDS = {"X710201": "STRIDE", "X710202": "KESTREL", "X710203": "EASYWALK"}
OTHER = {"X710298", "X710299"}  # other brand, don't know
CODES = {ANY} | set(PAIRS) | set(BRANDS) | OTHER
YES = {"1", "1.0", "Y", "YES", "TRUE"}

s = survey.copy()
s["rid_txt"] = s["RESPID"].str.strip()
s["kw"] = s["KEYWORD"].str.strip().str.upper()
s["val"] = s["VALUE"].str.strip().str.upper()
is_number = s["rid_txt"].str.fullmatch(r"\d+(\.0+)?")
s["rid"] = pd.to_numeric(s["rid_txt"].where(is_number), errors="coerce").astype("Int64")
s["pairs_n"] = s["kw"].map(PAIRS)

bad = []  # (kind of bad record, rows, what I did)

# rows that can be fixed
bad.append(("RESPID with spaces around it", int((s["RESPID"] != s["rid_txt"]).sum()), "fixed: spaces removed"))
bad.append(("RESPID written as a decimal (10000101.0)", int(s["rid_txt"].str.fullmatch(r"\d+\.0+").sum()), "fixed: read as integer"))
bad.append(("RESPID with leading zeros", int(s["rid_txt"].str.fullmatch(r"0\d+(\.0+)?").sum()), "fixed: read as integer"))
bad.append(("KEYWORD in lower case", int((s["KEYWORD"] != s["KEYWORD"].str.upper()).sum()), "fixed: upper case"))
bad.append(("KEYWORD with a trailing space", int((s["KEYWORD"] != s["KEYWORD"].str.strip()).sum()), "fixed: space removed"))
bad.append(("yes written as 1.0, y, Y, yes or TRUE", int((s["val"].isin(YES) & (s["VALUE"] != "1")).sum()), "fixed: counted as yes"))

# rows that must go (each row is counted under the first problem it has)
keep = pd.Series(True, index=s.index)


def drop(kind, mask, action="dropped"):
    hit = keep & mask
    bad.append((kind, int(hit.sum()), action))
    keep[hit] = False


drop("header line repeated inside the file, or the TOTAL ROWS footer", ~is_number)
drop("id of a panel member under 18", s["rid"].isin(persons - adults))
drop("id that is not a person in the panel", ~s["rid"].isin(adults))
drop("code from another survey (X880140, X880141, X880112, X902001)", ~s["kw"].isin(CODES))
drop("a no answer (0, 0.0, N, no) or an empty VALUE", ~s["val"].isin(YES))
drop("same person and keyword twice", s[keep].duplicated(["rid", "kw"]).reindex(s.index, fill_value=False))
pair_rows = s[keep & s["pairs_n"].notna()].sort_values("pairs_n", ascending=False)
people_two_counts = int((pair_rows["rid"].value_counts() > 1).sum())
drop("second, lower pair count of the same person", pair_rows.duplicated("rid").reindex(s.index, fill_value=False),
     "dropped: kept the higher count")

clean = s[keep]

# one row per adult panel member: buyer, pairs (0, 1, 2, 3+) and the three brands
people = pd.DataFrame(index=sorted(adults))
people.index.name = "respondent_id"
people["buyer"] = people.index.isin(clean["rid"])  # any yes answer: Any, a pair count or a brand
people["pairs"] = clean.groupby("rid")["pairs_n"].max().reindex(people.index).fillna(0).astype(int)
for code, name in BRANDS.items():
    people[name] = people.index.isin(clean.loc[clean["kw"] == code, "rid"])

print("Survey data rows:", int(is_number.sum()))  # without the 2 repeated headers and the footer
print("Clean yes answers:", len(clean))
print("Adult panel members:", len(people))
print("Non-buyers:", int((~people["buyer"]).sum()))
print("People with two pair counts:", people_two_counts)
display(people.assign(pairs=people["pairs"].map({0: "0", 1: "1", 2: "2", 3: "3+"})).head(10))

# table: kind of bad record, rows, what you did
display(pd.DataFrame(bad, columns=["kind of bad record", "rows", "what I did"]))

# table: KEYWORD, your count, layout SAMPLE
answers = survey_layout[survey_layout["RECTYPE"] == "B"][["KEYWORD", "Text", "SAMPLE"]].copy()
answers["SAMPLE"] = answers["SAMPLE"].astype(int)
my_count = {ANY: int(people["buyer"].sum())}  # every buyer, also those without the Any row
my_count.update({code: int((people["pairs"] == n).sum()) for code, n in PAIRS.items()})
my_count.update({code: int(people[name].sum()) for code, name in BRANDS.items()})
my_count.update({code: clean.loc[clean["kw"] == code, "rid"].nunique() for code in OTHER})
answers["my_count"] = answers["KEYWORD"].map(my_count)
answers["difference"] = answers["my_count"] - answers["SAMPLE"]
display(answers[["KEYWORD", "Text", "my_count", "SAMPLE", "difference"]].reset_index(drop=True))
no_any_row = people["buyer"] & ~people.index.isin(clean.loc[clean["kw"] == ANY, "rid"])
print("Buyers without an Any row:", int(no_any_row.sum()))

# the file as delivered: every row is a yes, every RESPID string is a different person
raw_kestrel = survey.loc[survey["KEYWORD"] == "X710202", "RESPID"].nunique()
raw_buyers = survey.loc[is_number, "RESPID"].nunique()  # without the header and footer lines
print("Kestrel buyers, as delivered vs clean (n):", raw_kestrel, "vs", int(people["KESTREL"].sum()))
print("Share of adults who are buyers, as delivered vs clean:", round(raw_buyers / len(people), 3), "vs",
      round(people["buyer"].mean(), 3))

Survey data rows: 18346
Clean yes answers: 15397
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,buyer,pairs,STRIDE,KESTREL,EASYWALK
respondent_id,,,,,
10000101,True,3+,False,False,False
10000102,True,2,False,False,True
10000201,True,1,True,False,False
10000301,True,1,False,False,True
10000302,False,0,False,False,False
10000401,True,1,False,False,False
10000501,False,0,False,False,False
10000502,True,1,True,False,False
10000601,True,3+,True,False,False


,kind of bad record,rows,what I did
0,RESPID with spaces around it,573,fixed: spaces removed
1,RESPID written as a decimal (10000101.0),533,fixed: read as integer
2,RESPID with leading zeros,349,fixed: read as integer
3,KEYWORD in lower case,734,fixed: upper case
4,KEYWORD with a trailing space,370,fixed: space removed
5,"yes written as 1.0, y, Y, yes or TRUE",1194,fixed: counted as yes
6,"header line repeated inside the file, or the T...",3,dropped
7,id of a panel member under 18,273,dropped
8,id that is not a person in the panel,150,dropped
9,"code from another survey (X880140, X880141, X8...",738,dropped


,KEYWORD,Text,my_count,SAMPLE,difference
0,X710100,Any,4779,4779,0
1,X710101,1 pair,1810,1810,0
2,X710102,2 pairs,1308,1308,0
3,X710103,3 or more pairs,1661,1661,0
4,X710201,STRIDE,1563,1563,0
5,X710202,KESTREL,354,354,0
6,X710203,EASYWALK,1050,1050,0
7,X710298,OTHER BRAND,2739,2739,0
8,X710299,DON'T KNOW,371,371,0


Buyers without an Any row: 238
Kestrel buyers, as delivered vs clean (n): 495 vs 354
Share of adults who are buyers, as delivered vs clean: 0.885 vs 0.597


**Answer:** The vendor promised one yes answer per person and keyword, only codes from the layout and only adult panel members. None of that held.

Some problems could be fixed. Ids came with spaces around them (573 rows), as decimals like `10000101.0` (533) or with leading zeros (349). Keywords were in lower case (734) or had a trailing space (370). And yes was not always `1`: in 1,194 rows it was `1.0`, `y`, `Y`, `yes` or `TRUE`.

Other rows I had to drop. Three lines were not data at all: the header shows up twice in the middle of the file, and there is a `TOTAL ROWS` footer at the end. Panel members under 18 account for 273 rows, and 150 rows have ids that are not persons in the panel. Another 738 rows use codes from other surveys (`X880140`, `X880141`, `X880112`, `X902001`). In 742 rows the answer is a no (`0`, `0.0`, `N`, `no`) or VALUE is empty, although the file should only have yes answers. There were also 933 duplicates of the same person and keyword. For the 113 people with two pair counts I kept the higher one.

That leaves 15,397 yes answers, and all nine codes match SAMPLE in the layout. For `Any` (4,779) this works only if the 238 people with a pair count or a brand but no `Any` row are counted as buyers, which is the lab's rule.

If I took the file as delivered, with every row a yes and every RESPID string a separate person, Kestrel would get 495 buyers instead of 354, about 40% too many. The share of adults who buy athletic shoes would be 0.885 instead of 0.597, because the same person written two ways, children, people outside the panel and the no answers would all count as buyers.

## Task 1.1 — Four definitions

In [5]:
buyers = set(people.index[people["buyer"]])
heavy = set(people.index[people["pairs"] == 3])
kestrel = set(people.index[people["KESTREL"]])
a25_54 = set(age[(age >= 25) & (age <= 54)].index)

audiences = {
    "Adults 18+": adults,
    "Shoe buyers": buyers,
    "Heavy buyers 3+": heavy,
    "Kestrel buyers": kestrel,
    "Adults 25-54": a25_54,
}
aud_size = {name: size(ids) for name, ids in audiences.items()}

print("Adults 18+ (M):", round(aud_size["Adults 18+"] / 1e6, 1))
print("Shoe buyers (M):", round(aud_size["Shoe buyers"] / 1e6, 1))
print("Heavy buyers, 3+ pairs (M):", round(aud_size["Heavy buyers 3+"] / 1e6, 1))
print("Kestrel buyers (M):", round(aud_size["Kestrel buyers"] / 1e6, 1))
print("Adults 25-54 (M):", round(aud_size["Adults 25-54"] / 1e6, 1))
print("Kestrel buyers in panel (n):", len(kestrel))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [6]:
groups = {}
for sx in ["M", "W"]:
    for name, lo, hi in [("18-34", 18, 34), ("35-54", 35, 54), ("55+", 55, 200)]:
        groups[f"{sx}{name}"] = set(age[(age >= lo) & (age <= hi)].index) & set(sex[sex == sx].index)

share = pd.DataFrame(index=list(groups), columns=list(audiences), dtype=float)
for g, g_ids in groups.items():
    for a, a_ids in audiences.items():
        share.loc[g, a] = size(a_ids & g_ids) / aud_size[a]
index_vs_adults = share.div(share["Adults 18+"], axis=0) * 100

# table: audience x group, share and index vs adults
print("Share of each audience in each group (0-1):")
display(share.round(3))
print("Index vs adults 18+ (100 = same as adults):")
display(index_vs_adults.round(1))

Share of each audience in each group (0-1):


,Adults 18+,Shoe buyers,Heavy buyers 3+,Kestrel buyers,Adults 25-54
M18-34,0.144,0.140,0.111,0.118,0.174
M35-54,0.160,0.158,0.138,0.189,0.326
M55+,0.185,0.186,0.189,0.126,0.000
W18-34,0.144,0.140,0.139,0.181,0.174
W35-54,0.160,0.178,0.226,0.265,0.326
W55+,0.208,0.197,0.196,0.121,0.000


Index vs adults 18+ (100 = same as adults):


,Adults 18+,Shoe buyers,Heavy buyers 3+,Kestrel buyers,Adults 25-54
M18-34,100.0,97.4,77.1,82.1,121.2
M35-54,100.0,99.2,86.8,118.6,204.1
M55+,100.0,100.8,102.2,68.2,0.0
W18-34,100.0,97.6,97.0,126.0,121.2
W35-54,100.0,111.4,141.9,165.8,204.1
W55+,100.0,94.6,94.0,58.0,0.0


**Answer:** I would give the planner heavy buyers (3+ pairs) for Kestrel. This is how the client itself defines the target, and the group is big enough to plan with: 49.6M people, about a fifth of adults, with 1,661 panelists behind it. Its profile is also clearly different from adults (women 35-54 have an index of 142, men 18-34 only 77). Kestrel buyers are closer to the brand, but with 10.0M people and only 354 panelists there are too few of them to rate programs, as Task 2.3 shows. Adults 25-54 is easy to buy. The problem is that it covers half of all adults (123.7M), so most people in it are not the target, and the age cut leaves out the 55+ heavy buyers, who make up 39% of the target.

"Athletic-shoe buyers" look almost like all adults because 59% of adults bought at least one pair in a year. A group that includes most of the population cannot be very different from it, so its indexes only go from 94.6 to 111.4.

The fused data has a limit, though. Buying was attached to each panelist only through age, sex and income, so we cannot tell whether the people who watch a program buy more shoes than others of the same age, sex and income. Within one demographic cell there is no link at all between what people watch and what they buy. A program's heavy-buyer index just repeats its demographic mix; it does not show that its viewers actually buy more running shoes.

## Task 2.1 — Programs by audience

In [7]:
def person_minutes(tel):
    """Minutes each person (R, L) viewed each telecast, on its network, with that day's weight."""
    m = tel[["telecast_id", "network_code", "broadcast_date", "start_ts", "end_ts"]].merge(
        v[["respondent_id", "network_code", "bc_date", "start_ts", "end_ts"]],
        left_on=["network_code", "broadcast_date"], right_on=["network_code", "bc_date"],
        suffixes=("", "_v"))
    m["minutes"] = (m["end_ts_v"].clip(upper=m["end_ts"])
                    - m["start_ts_v"].clip(lower=m["start_ts"])).dt.total_seconds() / 60
    m = m[m["minutes"] > 0]
    m = m.groupby(["telecast_id", "broadcast_date", "respondent_id"], as_index=False)["minutes"].sum()
    m = m.merge(dw_p[["respondent_id", "broadcast_date", "weight"]], on=["respondent_id", "broadcast_date"])
    m["wmin"] = m["weight"] * m["minutes"]
    return m


def rate(pm, tel, ids):
    """Average-minute rating (0-100) and viewers in an average minute, per telecast, for one audience."""
    wmin = pm[pm["respondent_id"].isin(ids)].groupby("telecast_id")["wmin"].sum()
    viewers = tel["telecast_id"].map(wmin).fillna(0) / tel["duration_min"]
    rating = viewers / tel["broadcast_date"].map(daily_size(ids)) * 100
    return rating, viewers


# price of a :30 for each telecast (Lab 2 rules: a program row wins over time rows)
rc = rate_card.copy()
is_prog_row = rc["program_name"].notna() & (rc["program_name"].astype(str).str.strip() != "")
rc_prog = rc[is_prog_row].drop_duplicates(["network_code", "program_name"])
rc_time = rc[~is_prog_row].copy()


def bc_minutes(hhmm):
    h, m = map(int, str(hhmm).split(":"))
    return h * 60 + m - 360


rc_time["start_min"] = rc_time["start_time"].map(bc_minutes)
rc_time["end_min"] = rc_time["end_time"].map(bc_minutes)
rc_time = rc_time.explode("weekdays")
rc_time["weekday"] = rc_time["weekdays"].astype(int)


def price_30s(tel):
    """Rate-card price of one :30 in each telecast (Series by telecast_id)."""
    t = tel[["telecast_id", "network_code", "program_name", "start_ts"]].copy()
    by_prog = t.merge(rc_prog[["network_code", "program_name", "cost_30s_usd"]],
                      on=["network_code", "program_name"], how="left").set_index("telecast_id")["cost_30s_usd"]
    bc = t["start_ts"] - pd.Timedelta(hours=6)
    t["weekday"] = bc.dt.dayofweek
    t["minute"] = bc.dt.hour * 60 + bc.dt.minute
    tm = t.merge(rc_time[["network_code", "weekday", "start_min", "end_min", "cost_30s_usd"]],
                 on=["network_code", "weekday"])
    tm = tm[(tm["minute"] >= tm["start_min"]) & (tm["minute"] < tm["end_min"])]
    by_time = tm.drop_duplicates("telecast_id").set_index("telecast_id")["cost_30s_usd"]
    return by_prog.fillna(by_time)


# the eight programs, October telecasts in daypart Prime
eight = [("NFL Football", "NBC"), ("NFL Football", "ABC"), ("Dancing with the Stars", "ABC"),
         ("NCIS", "CBS"), ("Hannity", "FNC"), ("Unsellable Houses", "HGTV"),
         ("Halloween Wars", "FOOD"), ("Grotesquerie", "FX")]
prime = prog[(prog["daypart"] == "Prime") & (prog["broadcast_date"] >= first_day)
             & (prog["broadcast_date"] <= last_day)]
pick = pd.concat([prime[(prime["program_name"].str.lower() == name.lower()) & (prime["network_code"] == net)]
                  for name, net in eight])
pick = pick.copy()
pick["label"] = pd.Categorical(pick["label"], categories=pick["label"].unique(), ordered=True)

pm8 = person_minutes(pick)
pick["adults_rating"], pick["adult_viewers"] = rate(pm8, pick, adults)
pick["target_rating"], pick["target_viewers"] = rate(pm8, pick, heavy)
pick["kestrel_rating"], _ = rate(pm8, pick, kestrel)
pick["cost_30s"] = pick["telecast_id"].map(price_30s(pick))
print("Telecasts without a price:", int(pick["cost_30s"].isna().sum()))

prg = pick.groupby("label", observed=True).agg(
    telecasts=("telecast_id", "count"),
    adults_rating=("adults_rating", "mean"),
    target_rating=("target_rating", "mean"),
    adult_viewers=("adult_viewers", "mean"),
    target_viewers=("target_viewers", "mean"),
    kestrel_rating=("kestrel_rating", "mean"),
    cost_30s=("cost_30s", "mean"),
)
prg["composition"] = prg["target_viewers"] / prg["adult_viewers"]
prg["index"] = prg["target_rating"] / prg["adults_rating"] * 100
prg["adults_cpm"] = prg["cost_30s"] / prg["adult_viewers"] * 1000
prg["target_cpm"] = prg["cost_30s"] / prg["target_viewers"] * 1000
prg["kestrel_index"] = prg["kestrel_rating"] / prg["adults_rating"] * 100

# table: 8 programs, the columns of Task 2.1
t21 = pd.DataFrame({
    "telecasts": prg["telecasts"],
    "adults_rating": prg["adults_rating"].round(2),
    "target_rating": prg["target_rating"].round(2),
    "target_viewers_k": (prg["target_viewers"] / 1e3).round(1),
    "composition": prg["composition"].round(3),
    "index": prg["index"].round(1),
    "cost_30s_usd": prg["cost_30s"].round(0).astype(int),
    "adults_cpm_usd": prg["adults_cpm"].round(0).astype(int),
    "target_cpm_usd": prg["target_cpm"].round(0).astype(int),
})
display(t21)
print("Target universe (M):", round(aud_size["Heavy buyers 3+"] / 1e6, 1))

Telecasts without a price: 0


,telecasts,adults_rating,target_rating,target_viewers_k,composition,index,cost_30s_usd,adults_cpm_usd,target_cpm_usd
label,,,,,,,,,
Sunday Night Football (NBC),4,5.82,5.79,2865.0,0.195,99.3,487200,33,170
Monday Night Football (ABC),3,8.93,8.36,4149.1,0.184,93.6,723600,32,174
Dancing with the Stars (ABC),9,3.25,3.14,1564.2,0.191,96.8,158600,19,101
NCIS (CBS),3,2.21,2.43,1206.3,0.216,110.1,178800,32,148
Hannity (FNC),22,1.09,0.84,416.9,0.152,77.1,39900,15,96
Unsellable Houses (HGTV),13,0.50,0.64,317.2,0.249,126.8,12900,10,41
Halloween Wars (FOOD),9,0.28,0.33,161.9,0.228,116.2,9700,14,60
Grotesquerie (FX),25,0.26,0.25,124.2,0.188,95.9,7592,11,61


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [8]:
print("Highest index (program):", prg["index"].idxmax())
print("Most target viewers (program):", prg["target_viewers"].idxmax())
print("Lowest target CPM (program):", prg["target_cpm"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** Unsellable Houses (HGTV) has the highest index (126.8), and it also has the lowest target CPM (USD 41). The most target viewers, though, come from Monday Night Football (ABC): 4.15M heavy buyers in an average minute, while Unsellable Houses has 317 thousand. So here index and price lead to the same small cable show, and volume leads somewhere else. To reach as many heavy buyers as one Monday Night Football spot does, we would need about 13 Unsellable Houses spots.

A client who wants "only the highest index" should hear the same thing as in the shampoo example from Lecture 5: an index tells you how concentrated the audience is, not how big it is. Unsellable Houses has an index of 127 but a rating of only 0.64, so it reaches few people, and even at that index only 25% of its adult viewers are heavy buyers. If we bought just the top-index program, we would get a small, cheap audience and run out of spots long before the budget was spent. I would build the plan on target CPM (the cost of reaching one thousand heavy buyers) and look at index mostly to compare programs with similar CPMs, then add big programs where the client needs reach.

## Task 2.3 — Kestrel buyers

In [9]:
# panelists behind each index: target panel members who watched any part of the telecast, on average
def panelists(ids):
    n = pm8[pm8["respondent_id"].isin(ids)].groupby("telecast_id")["respondent_id"].nunique()
    return pick["telecast_id"].map(n).fillna(0).groupby(pick["label"], observed=True).mean()


# table: program, heavy-buyer index, Kestrel index
t23 = pd.DataFrame({
    "heavy_buyer_index": prg["index"].round(1),
    "kestrel_index": prg["kestrel_index"].round(1),
    "heavy_buyer_panelists_per_telecast": panelists(heavy).round(1),
    "kestrel_panelists_per_telecast": panelists(kestrel).round(1),
})
display(t23)
print("Heavy buyers in panel (n):", len(heavy), "| Kestrel buyers in panel (n):", len(kestrel))

,heavy_buyer_index,kestrel_index,heavy_buyer_panelists_per_telecast,kestrel_panelists_per_telecast
label,,,,
Sunday Night Football (NBC),99.3,98.5,329.0,75.2
Monday Night Football (ABC),93.6,85.9,292.7,69.3
Dancing with the Stars (ABC),96.8,82.7,122.6,23.4
NCIS (CBS),110.1,56.5,86.3,11.0
Hannity (FNC),77.1,42.6,15.9,2.1
Unsellable Houses (HGTV),126.8,76.9,25.4,4.0
Halloween Wars (FOOD),116.2,95.2,13.7,2.8
Grotesquerie (FX),95.9,129.6,11.4,3.1


Heavy buyers in panel (n): 1661 | Kestrel buyers in panel (n): 354


**Answer:** The Kestrel index jumps a lot more: from 42.6 on Hannity to 129.6 on Grotesquerie, while the heavy-buyer index stays between 77.1 and 126.8. Mostly this is a sample size problem. The panel has only 354 Kestrel buyers (against 1,661 heavy buyers), and on the cable programs an average telecast is watched by just 2 to 4 of them, so one person more or less moves the index by 25-50%. Some of the difference is probably real, because Kestrel buyers skew more to women 35-54 (index 166) and less to people 55+ (index 58-68), and an older program like Hannity should come out low for them. On the small programs, though, we cannot tell this apart from noise.

I would trust the heavy-buyer index for the four big programs, where 86-329 heavy-buyer panelists stand behind each telecast, and use it with care for the cable shows and Hannity, where there are only 11-25. The Kestrel index I would only use for the two football games and Dancing with the Stars, where each telecast has 23-75 Kestrel panelists watching. As a check, I would first compute the index separately for the two halves of October (or for every telecast) to see if it stays the same, and add a standard error to it. It would also help to compare it with the index we would expect from Kestrel's age, sex and income profile alone. Since the data is fused, a program's Kestrel index cannot really be more accurate than that expected value anyway.

## Task 3.1 — Weights

In [10]:
# ACR sessions, Lab 3 rules: US Eastern time, broadcast day from 06:00, 1-minute edit rule
acr = acr_sessions.copy()
acr["start_et"] = pd.to_datetime(acr["start_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["end_et"] = pd.to_datetime(acr["end_utc"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
acr["start_et"] = acr["start_et"].astype("datetime64[ns]")
acr["end_et"] = acr["end_et"].astype("datetime64[ns]")
acr["bc_date"] = (acr["start_et"] - pd.Timedelta(hours=6)).dt.normalize()
acr = acr[(acr["end_et"] - acr["start_et"]).dt.total_seconds() >= 60]

# a TV is matched if its hashed_ip belongs to exactly one household
hh_per_ip = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
one_ip = hh_per_ip[hh_per_ip == 1].index
hh_one = hh_attributes[hh_attributes["hashed_ip"].isin(one_ip)].drop_duplicates("hashed_ip").set_index("hashed_ip")
tv_ip = acr_sessions.groupby("tv_id")["hashed_ip"].first()
tvs = tv_ip[tv_ip.isin(one_ip)].to_frame()
tvs["head_age"] = np.where(tvs["hashed_ip"].map(hh_one["head_age_band"]) == "65+", "65+", "Under 65")
tvs["income_band"] = tvs["hashed_ip"].map(hh_one["income_band"])

# matched TVs in-tab each day (at least one session after the edit rule), and their cell weight
uni = universe.copy()
uni["head_age"] = uni["head_age"].str.strip()
uni["income_band"] = uni["income_band"].str.strip()
intab = acr.loc[acr["tv_id"].isin(tvs.index), ["bc_date", "tv_id"]].drop_duplicates()
intab = intab.merge(tvs[["head_age", "income_band"]], left_on="tv_id", right_index=True)
intab["n_cell"] = intab.groupby(["bc_date", "head_age", "income_band"])["tv_id"].transform("count")
intab = intab.merge(uni[["head_age", "income_band", "tv_households"]], on=["head_age", "income_band"], how="left")
intab["weight"] = intab["tv_households"] / intab["n_cell"]
print("TVs without a universe cell:", int(intab["weight"].isna().sum()))

w_sum = intab.groupby("bc_date")["weight"].sum()
print("TVs matched to one household:", len(tvs))
print("TVs with more than one hashed_ip:", int((acr_sessions.groupby("tv_id")["hashed_ip"].nunique() > 1).sum()))
print("Head age band of matched TVs:", tvs["hashed_ip"].map(hh_one["head_age_band"]).value_counts(dropna=False).to_dict())
print("Weights sum per day (M):", round(w_sum.mean() / 1e6, 1))
print("Universe (M):", round(uni["tv_households"].sum() / 1e6, 1),
      "| every day adds up:", bool(np.allclose(w_sum, uni["tv_households"].sum())))

# ratings of the reference telecasts from matched TVs, unweighted and weighted
ref = reference_ratings.merge(prog[["telecast_id", "label", "network_code", "broadcast_date", "start_ts", "end_ts"]],
                              on="telecast_id")
live = acr[acr["session_type"] == "live"]
rows, by_cell = [], []
for _, t in ref.iterrows():
    day = intab[intab["bc_date"] == t["broadcast_date"]]
    s_ = live[(live["network_code"] == t["network_code"]) & live["tv_id"].isin(day["tv_id"])]
    overlap = (s_["end_et"].clip(upper=t["end_ts"]) - s_["start_et"].clip(lower=t["start_ts"])).dt.total_seconds() / 60
    watched = day["tv_id"].isin(s_.loc[overlap >= 1, "tv_id"])
    rows.append({"telecast": t["label"],
                 "date": t["broadcast_date"].strftime("%Y-%m-%d"),
                 "unweighted": watched.mean() * 100,
                 "weighted": day.loc[watched, "weight"].sum() / day["weight"].sum() * 100,
                 "reference": t["hh_rating"]})
    by_cell.append((day.assign(watched=watched).groupby(["head_age", "income_band"])["watched"].mean() * 100)
                   .rename(t["network_code"]))
t31 = pd.DataFrame(rows)

# table: telecast, unweighted, weighted, reference
display(t31.round(2))
gap_unw = (t31["reference"] - t31["unweighted"]).abs().mean()
gap_w = (t31["reference"] - t31["weighted"]).abs().mean()
print("Mean gap unweighted (points):", round(gap_unw, 2))
print("Mean gap weighted (points):", round(gap_w, 2))
print("Share of gap closed:", round((gap_unw - gap_w) / gap_unw, 3))

TVs without a universe cell: 0
TVs matched to one household: 19607
TVs with more than one hashed_ip: 0
Head age band of matched TVs: {'35-49': 6718, '50-64': 5151, '18-34': 3930, '65+': 3808}
Weights sum per day (M): 125.8
Universe (M): 125.8 | every day adds up: True


,telecast,date,unweighted,weighted,reference
0,MLB Postseason (FOX),2024-10-25,17.03,17.76,25.86
1,Sunday Night Football (NBC),2024-10-27,22.62,23.75,33.14
2,NCIS (CBS),2024-10-14,5.50,5.80,10.56
3,Hannity (FNC),2024-10-16,1.04,1.35,2.30
4,Unsellable Houses (HGTV),2024-10-16,2.12,2.08,3.69
5,Halloween Baking Championship (FOOD),2024-10-14,0.95,0.90,1.63
6,Grotesquerie (FX),2024-10-16,2.17,2.11,3.53
7,The Real Housewives of New York City (BRAVO),2024-10-15,0.71,0.64,1.23


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


In [11]:
# whose homes the footprint over- and under-represents: matched in-tab TVs vs the universe
bands = ["Under $25k", "$25k-$50k", "$50k-$75k", "$75k-$100k", "$100k-$150k", "$150k+"]
cell_order = pd.MultiIndex.from_product([["Under 65", "65+"], bands], names=["head_age", "income_band"])
foot = intab.groupby(["head_age", "income_band"]).size() / len(intab)
uni_share = uni.set_index(["head_age", "income_band"])["tv_households"] / uni["tv_households"].sum()
cells = pd.DataFrame({"footprint_share": foot, "universe_share": uni_share}).reindex(cell_order).fillna(0)
cells["index"] = cells["footprint_share"] / cells["universe_share"] * 100
display(cells.round(3))
display(cells.groupby(level="head_age", sort=False)[["footprint_share", "universe_share"]].sum().round(3))

# unweighted rating of the matched TVs in each cell, for the eight reference telecasts
print("Rating by cell (0-100), columns = network of the reference telecast:")
display(pd.concat(by_cell, axis=1).reindex(cell_order).round(2))

footprint_share  universe_share    index
head_age income_band                                          
Under 65 Under $25k             0.072           0.084   86.009
         $25k-$50k              0.088           0.100   88.069
         $50k-$75k              0.115           0.108  105.932
         $75k-$100k             0.127           0.092  137.837
         $100k-$150k            0.176           0.139  127.138
         $150k+                 0.228           0.194  117.453
65+      Under $25k             0.028           0.060   46.187
         $25k-$50k              0.033           0.062   54.038
         $50k-$75k              0.031           0.048   64.539
         $75k-$100k             0.029           0.034   85.585
         $100k-$150k            0.036           0.039   90.899
         $150k+                 0.037           0.040   92.984

,footprint_share,universe_share
head_age,,
Under 65,0.806,0.717
65+,0.194,0.283


Rating by cell (0-100), columns = network of the reference telecast:


FOX    NBC    CBS   FNC  HGTV  FOOD    FX  BRAVO
head_age income_band                                                    
Under 65 Under $25k   11.85  15.94   4.02  0.35  1.76  0.56  1.27   0.42
         $25k-$50k    12.16  16.50   3.30  0.23  1.33  0.93  2.03   0.64
         $50k-$75k    13.93  18.96   4.63  0.45  2.09  0.67  1.74   0.76
         $75k-$100k   13.82  18.43   4.38  0.40  2.01  1.12  2.01   0.40
         $100k-$150k  15.61  21.03   5.24  0.46  2.11  0.78  2.17   0.84
         $150k+       17.96  22.02   5.46  0.56  2.44  1.23  2.98   1.19
65+      Under $25k   24.44  33.89   8.52  4.44  1.11  0.37  1.67   0.19
         $25k-$50k    24.16  32.72   7.65  4.28  1.99  0.61  2.45   0.00
         $50k-$75k    25.74  36.39  10.82  3.44  2.46  0.33  1.15   0.00
         $75k-$100k   25.48  34.32   9.71  3.29  2.25  1.21  1.91   0.00
         $100k-$150k  23.31  35.54   6.76  3.74  3.31  2.01  3.17   0.86
         $150k+       28.42  36.34   8.47  2.05  2.60  1.09  1.37   0.82

## Task 3.2 — What weighting cannot fix
**Answer:** After weighting, the telecasts that older homes watch went up: Hannity from 1.04 to 1.35, Sunday Night Football from 22.62 to 23.75, the World Series game from 17.03 to 17.76 and NCIS from 5.50 to 5.80. The cell table explains it. Homes with a head aged 65+ watch these programs much more than younger homes: NBC and FOX get roughly twice the rating there (NBC about 33-36 against 16-22), and Fox News several times more (2-4 against less than 0.6). The footprint also has too few of these homes: 0.194 against 0.283 in the universe, and the poorer 65+ homes have an index of only 46-54. So these homes get bigger weights. Unsellable Houses, Halloween Baking Championship, Grotesquerie and Real Housewives went down by 0.04-0.07 points, since older homes watch them about as much as younger ones or less (for Bravo, half of the 65+ cells are at 0).

Still, weighting closed only 7.5% of the gap (from 3.73 points to 3.44), and the weighted ratings are only 52-72% of the reference. The reason is that weighting can only correct the mix of homes, and only on the variables we weight by. It does nothing about what the device cannot see (Lab 3): the feed measures one smart TV per home, while a home has about 2.6 sets, and a big part of viewing time stays unknown. Who opted in is another problem (Lecture 4), because smart-TV homes in a cell may watch differently from the other homes in that cell. The TVs that did not match exactly one household are also simply lost.

Income changes almost nothing here, for two reasons. First, the footprint is off much more on age than on income, and the rating gap between the age groups is far bigger than between income bands (NBC gets 33-36 in 65+ homes but 16-22 in younger ones, while within an age group income moves it by up to 6 points). Second, the two corrections partly cancel out: under 65 the footprint has too many high-income homes (cell indexes 117-138 from 75k up), and these homes watch more, so weighting by income pulls ratings slightly down while weighting by age pushes them up.

## Task 4.1 — October predicts October

In [12]:
big5 = ["CBS", "NBC", "ABC", "FOX", "FNC"]
pt = prog[(prog["daypart"] == "Prime") & prog["network_code"].isin(big5)
          & (prog["broadcast_date"] >= first_day) & (prog["broadcast_date"] <= pd.Timestamp("2024-10-28"))].copy()
pm_pt = person_minutes(pt)
pt["p2"], _ = rate(pm_pt, pt, p2_plus)
pt["heavy"], _ = rate(pm_pt, pt, heavy)

hist = pt[pt["broadcast_date"] <= pd.Timestamp("2024-10-21")]
week = pt[pt["broadcast_date"] >= pd.Timestamp("2024-10-22")]
fc_mean = hist.groupby(["network_code", "program_name"])[["p2", "heavy"]].mean().add_prefix("fc_").reset_index()
fc = week.merge(fc_mean, on=["network_code", "program_name"])  # only programs that aired in history


def ape(actual, forecast):
    """|actual - forecast| / actual; telecasts with an actual of 0 are skipped (NaN)."""
    return ((actual - forecast).abs() / actual).where(actual > 0)


fc["err_p2"] = ape(fc["p2"], fc["fc_p2"])
fc["err_heavy"] = ape(fc["heavy"], fc["fc_heavy"])
worst = fc.loc[fc["err_p2"].idxmax()]

print("Telecasts forecast (n):", len(fc))
print("MAPE, P2+ (%):", round(fc["err_p2"].mean() * 100, 1))
print("MAPE, heavy buyers (%):", round(fc["err_heavy"].mean() * 100, 1))
print("Biggest miss, P2+ (program, date):", f"{worst['label']}, {worst['broadcast_date']:%Y-%m-%d}")
print("Skipped (actual = 0): P2+", int(fc["err_p2"].isna().sum()), "| heavy buyers", int(fc["err_heavy"].isna().sum()))
print("WAPE, P2+ (%):", round((fc["p2"] - fc["fc_p2"]).abs().sum() / fc["p2"].sum() * 100, 1))
print("WAPE, heavy buyers (%):", round((fc["heavy"] - fc["fc_heavy"]).abs().sum() / fc["heavy"].sum() * 100, 1))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27
Skipped (actual = 0): P2+ 0 | heavy buyers 0
WAPE, P2+ (%): 22.8
WAPE, heavy buyers (%): 24.6


## Task 4.2 — Why it missed

In [13]:
# table: the five biggest misses (program, date, forecast, actual)
top5 = fc.sort_values("err_p2", ascending=False).head(5)
display(pd.DataFrame({
    "program": top5["label"],
    "date": top5["broadcast_date"].dt.strftime("%Y-%m-%d"),
    "start": top5["start_ts"].dt.strftime("%H:%M"),
    "forecast_p2": top5["fc_p2"].round(2),
    "actual_p2": top5["p2"].round(2),
    "error_pct": (top5["err_p2"] * 100).round(1),
}).reset_index(drop=True))

# what aired on those networks on those days (from 12:00), and on the two FOX evenings with high history ratings
to_show = [(t_["network_code"], t_["broadcast_date"]) for _, t_ in top5.iterrows()]
to_show += [("FOX", pd.Timestamp("2024-10-01")), ("FOX", pd.Timestamp("2024-10-20"))]
for net, day in dict.fromkeys(to_show):
    grid = prog[(prog["network_code"] == net) & (prog["broadcast_date"] == day)
                & (prog["start_ts"] >= day + pd.Timedelta(hours=12)) & (prog["start_ts"] < day + pd.Timedelta(hours=25))]
    print(f"\n{net} on {day:%Y-%m-%d}:")
    display(grid[["start_ts", "end_ts", "program_name", "episode_name", "program_type", "is_live", "is_repeat"]]
            .reset_index(drop=True))

# what the big networks showed at 21:00 on October 1
oct1 = pd.Timestamp("2024-10-01")
print("\nBig networks at 21:00 on 2024-10-01:")
display(prog[(prog["broadcast_date"] == oct1) & prog["network_code"].isin(big5)
             & (prog["start_ts"] <= oct1 + pd.Timedelta(hours=21)) & (prog["end_ts"] > oct1 + pd.Timedelta(hours=21))]
        [["network_code", "start_ts", "end_ts", "program_name", "is_live"]].reset_index(drop=True))

# every October 1-28 prime telecast of the five programs: when it aired, what came before it (lead-in), rating
prog_sorted = prog.sort_values(["network_code", "start_ts"])
lead_in = prog_sorted.groupby("network_code")["program_name"].shift(1)
pt["lead_in"] = pt.index.map(lead_in)
for _, t in top5.iterrows():
    h = pt[(pt["network_code"] == t["network_code"]) & (pt["program_name"] == t["program_name"])]
    print(f"\n{t['label']}: history and forecast week")
    display(pd.DataFrame({
        "date": h["broadcast_date"].dt.strftime("%Y-%m-%d"),
        "weekday": h["broadcast_date"].dt.day_name(),
        "start": h["start_ts"].dt.strftime("%H:%M"),
        "minutes": h["duration_min"],
        "repeat": h["is_repeat"],
        "lead_in": h["lead_in"],
        "p2": h["p2"].round(2),
    }).reset_index(drop=True))

# panelists behind the two forecasts
print("Persons 2+ in-tab per day (n):",
      round(dw_p[dw_p["respondent_id"].isin(p2_plus)].groupby("broadcast_date")["respondent_id"].nunique().mean()))
print("Heavy buyers in-tab per day (n):",
      round(dw_p[dw_p["respondent_id"].isin(heavy)].groupby("broadcast_date")["respondent_id"].nunique().mean()))

,program,date,start,forecast_p2,actual_p2,error_pct
0,Krapopolis (FOX),2024-10-27,21:30,1.06,0.22,387.8
1,Universal Basic Guys (FOX),2024-10-27,20:30,0.68,0.26,162.2
2,Fire Country (CBS),2024-10-27,22:44,1.70,0.67,154.8
3,Accused (FOX),2024-10-22,21:00,0.95,0.37,154.6
4,What Would You Do? (ABC),2024-10-23,22:00,0.44,0.18,138.1



FOX on 2024-10-27:


,start_ts,end_ts,program_name,episode_name,program_type,is_live,is_repeat
0,2024-10-27 12:00:00,2024-10-27 13:00:00,FOX NFL Sunday,Week 8,talk,True,False
1,2024-10-27 13:00:00,2024-10-27 16:05:00,NFL Football,NFL Football: regional coverage (4 games),sports,True,False
2,2024-10-27 16:05:00,2024-10-27 19:15:00,NFL Football,NFL Football: regional coverage (2 games),sports,True,False
3,2024-10-27 19:15:00,2024-10-27 20:00:00,Local / Syndicated Programming,NaN,local,False,False
4,2024-10-27 20:00:00,2024-10-27 20:30:00,The Simpsons,Shoddy Heat,series,False,False
5,2024-10-27 20:30:00,2024-10-27 21:00:00,Universal Basic Guys,Fight or Flight,series,False,False
6,2024-10-27 21:00:00,2024-10-27 21:30:00,Bob's Burgers,For Whom the Doll Toes,series,False,False
7,2024-10-27 21:30:00,2024-10-27 22:00:00,Krapopolis,Mr. Boogens,series,False,False
8,2024-10-27 22:00:00,2024-10-27 22:30:00,Bob's Burgers,NaN,series,False,True
9,2024-10-27 22:30:00,2024-10-27 23:00:00,Krapopolis,NaN,series,False,True



CBS on 2024-10-27:


,start_ts,end_ts,program_name,episode_name,program_type,is_live,is_repeat
0,2024-10-27 12:00:00,2024-10-27 13:00:00,The NFL Today,Week 8,talk,True,False
1,2024-10-27 13:00:00,2024-10-27 16:10:00,NFL Football,New York Jets at New England Patriots,sports,True,False
2,2024-10-27 16:10:00,2024-10-27 16:25:00,Local / Syndicated Programming,NaN,local,False,False
3,2024-10-27 16:25:00,2024-10-27 19:35:00,NFL Football,Chicago Bears at Washington Commanders,sports,True,False
4,2024-10-27 19:35:00,2024-10-27 20:00:00,60 Minutes,Deportation | Sanctions | Surfmen,news,True,False
5,2024-10-27 20:00:00,2024-10-27 20:30:00,Poppa's House,NaN,series,False,True
6,2024-10-27 20:30:00,2024-10-27 20:44:00,Fire Country,NaN,series,False,True
7,2024-10-27 20:44:00,2024-10-27 21:44:00,Tracker,Bloodlines,series,False,False
8,2024-10-27 21:44:00,2024-10-27 22:44:00,The Equalizer,Haunted Heights,series,False,False
9,2024-10-27 22:44:00,2024-10-27 23:00:00,Fire Country,NaN,series,False,True



FOX on 2024-10-22:


... (16 outputs left out)

,date,weekday,start,minutes,repeat,lead_in,p2
0,2024-10-01,Tuesday,21:00,60,True,Murder in a Small Town,1.87
1,2024-10-08,Tuesday,21:00,60,False,Murder in a Small Town,0.61
2,2024-10-15,Tuesday,21:00,60,False,Murder in a Small Town,0.38
3,2024-10-22,Tuesday,21:00,60,False,Murder in a Small Town,0.37
4,2024-10-23,Wednesday,22:00,60,True,The Floor,2.14



What Would You Do? (ABC): history and forecast week


,date,weekday,start,minutes,repeat,lead_in,p2
0,2024-10-07,Monday,22:00,60,False,Dancing with the Stars,0.49
1,2024-10-16,Wednesday,22:00,60,False,Abbott Elementary,0.38
2,2024-10-23,Wednesday,22:00,60,False,Abbott Elementary,0.18


Persons 2+ in-tab per day (n): 9269
Heavy buyers in-tab per day (n): 1493


**Answer:** In all five cases the history the forecast was built on did not match the evening it had to predict. Krapopolis (FOX, October 27) was averaged over two different slots: two Wednesday 22:00 repeats after The Floor (1.76 and 1.93) and two Sunday 21:30 episodes after Bob's Burgers (0.26 and 0.30). A mean of 1.06 fits neither of them, and the actual 0.22 is a normal Sunday number. With Universal Basic Guys it is similar. Its one Sunday 20:30 airing in the history got 0.26, but on October 20 it aired at 20:00, right after FOX's late NFL game (Kansas City at San Francisco, until 19:35), and the football audience pushed it to 1.09. That one evening alone pulled the forecast up to 0.68. Fire Country (CBS, October 27) had its new episode in the history on Friday at 21:00 after S.W.A.T. (1.81 on October 18). On Sunday the 27th, CBS's late NFL game (Bears at Commanders) ran until 19:35, so 60 Minutes started late, the whole evening shifted, and a repeat of Fire Country was cut into two pieces of 14 and 16 minutes. The 22:44 piece got 0.67, while the forecast was 1.70. Accused (FOX, October 22) was losing viewers every week (0.61, then 0.38), but the first history telecast on October 1, a repeat, rated 1.87. That evening ABC, CBS, NBC and Fox News all showed the vice-presidential debate at 21:00, so FOX was the only one of the five big networks with something else on, which most likely explains the high number for a repeat. It put the mean at 0.95, far above the trend, and the actual 0.37 simply continued the decline. What Would You Do? (ABC, October 23) is only partly a lead-in story. Its history has one Monday airing after Dancing with the Stars (0.49) and one Wednesday airing after Abbott Elementary (0.38), so even in the same Wednesday slot it fell from 0.38 to 0.18. The stronger Monday lead-in explains part of the miss, and the rest is noise in a very small rating.

The heavy-buyer forecast misses more (MAPE 38.6% against 35.8%) because it is based on far fewer people: on an average day about 1,493 heavy buyers are in-tab, compared with 9,269 persons. So both the history mean and the actual rating are noisier, and as Lecture 6 says, the error grows when there are fewer panelists in the target.

I would change two things in the forecast. First, I would build it from the schedule and not just from the program name: use only telecasts in the same weekday and time slot, and take into account the lead-in, repeats, shortened pieces and known events such as sports overruns or Election Night on November 5. Second, for heavy buyers I would forecast the P2+ rating, which rests on many more people, and multiply it by the program's long-run heavy-buyer index, instead of averaging a few noisy heavy-buyer ratings.

## Task 5.1 — The plan

In [14]:
budget = 1_000_000
weeks_in_october = 31 / 7
cand = prg[["cost_30s", "target_viewers", "adult_viewers", "target_cpm", "telecasts"]].copy()
cand["cost_30s"] = cand["cost_30s"].round(0)
# at most two :30s per telecast, and as many telecasts as the program airs in a week
cand["max_spots"] = 2 * np.maximum(1, np.round(cand["telecasts"] / weeks_in_october)).astype(int)
# Halloween Wars and Grotesquerie both ended in October, so they cannot be bought in November
cand = cand.drop(index=["Halloween Wars (FOOD)", "Grotesquerie (FX)"])
# Dancing with the Stars has one new episode in the week of November 11-17 (November 12)
cand.loc["Dancing with the Stars (ABC)", "max_spots"] = 2
cand = cand.sort_values("target_cpm")
costs, viewers = cand["cost_30s"].tolist(), cand["target_viewers"].tolist()


def best_mix(max_spots):
    """Try every mix of whole spots within the limits; keep the one with the most target impressions in budget."""
    best_imp, best = -1, None
    for mix in product(*[range(m + 1) for m in max_spots]):
        cost = sum(n * c for n, c in zip(mix, costs))
        imp = sum(n * v for n, v in zip(mix, viewers))
        if cost <= budget and imp > best_imp:
            best_imp, best = imp, mix
    return pd.Series(best, index=cand.index)


cand["spots"] = best_mix(cand["max_spots"])
plan = cand[cand["spots"] > 0].copy()
plan["cost"] = plan["spots"] * plan["cost_30s"]
plan["target_impressions"] = plan["spots"] * plan["target_viewers"]
plan["adult_impressions"] = plan["spots"] * plan["adult_viewers"]

# table: program, spots, cost, target impressions
display(pd.DataFrame({
    "spots": plan["spots"],
    "cost_per_30s_usd": plan["cost_30s"].astype(int),
    "cost_usd": plan["cost"].astype(int),
    "target_impressions_M": (plan["target_impressions"] / 1e6).round(2),
    "target_cpm_usd": plan["target_cpm"].round(0).astype(int),
}))
plan_cost = plan["cost"].sum()
plan_target = plan["target_impressions"].sum()
print("Plan cost ($):", int(plan_cost))
print("Plan target impressions (M):", round(plan_target / 1e6, 1))
print("Plan target CPM ($):", round(plan_cost / plan_target * 1000))
print("Plan composition (heavy buyers / adults):", round(plan_target / plan["adult_impressions"].sum(), 3))

,spots,cost_per_30s_usd,cost_usd,target_impressions_M,target_cpm_usd
label,,,,,
Unsellable Houses (HGTV),6,12900,77400,1.90,41
Hannity (FNC),10,39900,399000,4.17,96
Dancing with the Stars (ABC),2,158600,317200,3.13,101
NCIS (CBS),1,178800,178800,1.21,148


Plan cost ($): 972400
Plan target impressions (M): 10.4
Plan target CPM ($): 93
Plan composition (heavy buyers / adults): 0.182


## Task 5.2 — Recommendation
**Answer:** For the week of November 11-17, right after the election week, I recommend 19 spots for USD 972,400: 6 in Unsellable Houses, 10 in Hannity, 2 in Dancing with the Stars and 1 in NCIS, for about 10.4M heavy-buyer impressions at a target CPM of USD 93. I tried every mix of whole spots within the budget and kept the one with the most heavy-buyer impressions, using two assumptions of my own (at most two :30s per telecast, and as many telecasts per week as each program had in October), corrected for what is known about November: Halloween Wars and Grotesquerie ended in October, and Dancing with the Stars has one episode that week, on November 12. The football games did not make it because they cost too much per heavy buyer (a CPM of USD 170-174). Hannity takes 41% of the money although it has the lowest index (77) and composition (0.152) of the eight programs, so we buy it for its low price, not for its audience fit. In the week after the election its rating may also differ from October, and the political news around it can be a brand-safety question for a sports brand, so the client should approve these spots or move some of the money to NCIS. Delivery is not certain: for heavy buyers our October forecast missed the total by 24.6% (WAPE) and single telecasts by 38.6% on average (MAPE), and a cable show like Unsellable Houses rests on only about 25 heavy-buyer panelists per telecast, so I would ask the networks for a guarantee in heavy-buyer impressions with make-goods. About 18% of the impressions reach heavy buyers, so roughly 82% of the money goes to people outside the target; for TV this is normal, since even the best program here gets only 25%. After the week I would check the delivered heavy-buyer impressions spot by spot in the panel and the reach and frequency among heavy buyers, because ten Hannity spots may hit the same people too often, and together with the client I would look at Kestrel's sales or brand lift.

## Bonus

**Answer:** I did not enter the Kaggle competition.